In [ ]:
from pathlib import Path
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 从当前工作目录逐级向上寻找公开项目根目录，不依赖本机盘符。
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current, *current.parents]
                     if (p / "data" / "analysis_base.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("未找到 data/analysis_base.csv，请从项目目录或其子目录运行 notebook。")

DATA_PATH = PROJECT_ROOT / "data" / "analysis_base.csv"
OUT_DIR = PROJECT_ROOT / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200)

CATS = ["医疗卫生设施数", "教育文化设施数", "体育休闲设施数",
        "公园景点设施数", "交通设施数", "政务社区设施数"]
TOTAL = "500米公共服务设施总数（个）"
KEY = ["计划年份", "小区名称"]

ana = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print(f"项目根：{PROJECT_ROOT}")
print(f"分析底表：{len(ana)} 行 × {ana.shape[1]} 列")


In [ ]:
# ===== 2. 区域差异：各区周边公共服务水平 =====
import seaborn as sns

SHORT = {c: c.replace('设施数', '') for c in CATS}
by_dist = ana.groupby('所在区').agg(
    小区数=(TOTAL, 'size'),
    设施总数中位=(TOTAL, 'median'),
    设施总数均值=(TOTAL, 'mean'),
    设施总数标准差=(TOTAL, 'std'),
).round(1)
for c, s in SHORT.items():
    by_dist[s + '中位'] = ana.groupby('所在区')[c].median()
by_dist = by_dist.sort_values('设施总数中位', ascending=False)

print("① 各区分项设施中位数（按设施总数中位降序）")
print(by_dist.to_string())

print(f"\n② 建成年代覆盖：{ana['建成年代'].notna().sum()} / {len(ana)} 行")
print(ana['建成年代'].value_counts().sort_index().to_string())

# ---------- 图 1：各区设施总数分布 ----------
order = by_dist.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.boxplot(data=ana, x='所在区', y=TOTAL, order=order, ax=axes[0],
            hue='所在区', palette='Set2', legend=False, showfliers=True)
axes[0].set_title('各区老旧小区周边 500m 公共服务设施总数分布', fontsize=13)
axes[0].set_xlabel(''); axes[0].set_ylabel('设施总数（个）')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=.3)

plot_df = by_dist.reset_index().melt(id_vars=['所在区', '小区数'],
                                     value_vars=[s + '中位' for s in SHORT.values()],
                                     var_name='类别', value_name='中位数')
plot_df['类别'] = plot_df['类别'].str.replace('中位', '', regex=False).str.replace('设施', '')
sns.barplot(data=plot_df, x='所在区', y='中位数', hue='类别', order=order, ax=axes[1])
axes[1].set_title('各区分项设施中位数构成', fontsize=13)
axes[1].set_xlabel(''); axes[1].set_ylabel('设施数中位（个）')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='', fontsize=8, ncol=2)
axes[1].grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '图1_区域差异.png'), dpi=140, bbox_inches='tight')
plt.show()

print("\n③ 2024 vs 2025 批次对比")
print(ana.groupby('计划年份')[TOTAL].agg(['size', 'median', 'mean']).round(1).to_string())

In [ ]:
# ===== 3. 空间分布格局（分级放大 · 解决离群点压缩）=====
# 痛点：溧水/江宁远郊小区距市中心 20~50km，与主城小区（>85% 在 9km 内）
#       同框时被压成一小团 → 改为「全市 → 主城 → 核心区」三级放大。
def hav_m(lng1, lat1, lng2, lat2):
    """Haversine 球面距离（米），支持标量/Series"""
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(np.subtract(lat2, lat1)), np.radians(np.subtract(lng2, lng1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

XJX = (118.7787, 32.0419)          # 新街口（市中心参照点）
LAT0 = 32.04
AR = 1 / np.cos(np.radians(LAT0))  # 1°经度 ≈ 94.3km，1°纬度 ≈ 111km → 比例修正
KM_LAT, KM_LNG = 1 / 111.0, 1 / (111.0 * np.cos(np.radians(LAT0)))

ana['距市中心_km'] = hav_m(ana['经度'], ana['纬度'], *XJX) / 1000

# ---------- 统计 ----------
from scipy import stats as st
print("① 距市中心距离 与 设施总数 的关系")
print(ana['距市中心_km'].describe().round(1).to_string())
rho, pv = st.spearmanr(ana['距市中心_km'], ana[TOTAL])
print(f"\nSpearman ρ = {rho:.3f}   p = {pv:.2e}")

ana['距市中心分层'] = pd.cut(ana['距市中心_km'], [0, 3, 6, 10, 20, 100],
                             labels=['<3km 核心', '3-6km', '6-10km', '10-20km', '>20km 外围'])
print("\n② 按距市中心分层")
print(ana.groupby('距市中心分层', observed=True)[TOTAL]
      .agg(['size', 'median', 'mean']).round(1).to_string())

# ---------- 图 2：三级放大 ----------
L_CITY, L_CORE = 9.0, 3.2          # 主城 / 核心区 半径阈值（km）
city = ana[ana['距市中心_km'] <= L_CITY]
core = ana[ana['距市中心_km'] <= L_CORE]
print(f"\n③ 绘图分层：全市 {len(ana)} → 主城(≤{L_CITY:g}km) {len(city)} "
      f"({len(city)/len(ana):.0%}) → 核心(≤{L_CORE:g}km) {len(core)}")

def lims(df_, pad_km):
    """按**等物理距离**加边距的经纬度范围（避免按度数加边距导致不等距留白）"""
    return (df_['经度'].min() - pad_km * KM_LNG, df_['经度'].max() + pad_km * KM_LNG,
            df_['纬度'].min() - pad_km * KM_LAT, df_['纬度'].max() + pad_km * KM_LAT)

def set_extent(ax, df_, pad_km):
    x0, x1, y0, y1 = lims(df_, pad_km)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_aspect(AR)                      # 经纬度等距，形状不失真
    ax.set_xlabel('经度'); ax.set_ylabel('纬度')
    return x0, x1, y0, y1

def ring(ax, r_km, txt=True):
    """以新街口为圆心的等距圈，提供空间尺度感"""
    th = np.linspace(0, 2 * np.pi, 361)
    ax.plot(XJX[0] + r_km * KM_LNG * np.cos(th), XJX[1] + r_km * KM_LAT * np.sin(th),
            ls=':', lw=.9, c='grey', zorder=0)
    if txt:
        ax.annotate(f'{r_km:g}km', (XJX[0], XJX[1] + r_km * KM_LAT),
                    textcoords='offset points', xytext=(3, 3), fontsize=7, color='grey')

def mark_center(ax, label=True):
    ax.plot(*XJX, marker='*', ms=16, c='crimson', zorder=6)
    if label:
        ax.annotate('新街口', XJX, textcoords='offset points', xytext=(7, 5),
                    color='crimson', fontsize=8.5, zorder=7)

palette = dict(zip(by_dist.index, sns.color_palette('Set2', len(by_dist))))

def scatter_district(ax, df_, k=1.0, base=16):
    for dn, g in df_.groupby('所在区'):
        ax.scatter(g['经度'], g['纬度'],
                   s=g[TOTAL] / ana[TOTAL].max() * 300 * k + base * k,
                   c=[palette[dn]], alpha=.7, edgecolors='white',
                   linewidths=.6, label=dn, zorder=3)

fig, axes = plt.subplots(2, 2, figsize=(14, 13.5), layout='constrained')

# -------- ① 全市概览（含离群点，用虚线框提示放大区）--------
ax = axes[0, 0]
scatter_district(ax, ana, .75)
ring(ax, 10, txt=False)
# 先固定范围，再在范围内标注，避免文字被边框裁切
x0, x1, y0, y1 = set_extent(ax, ana, 3.5)
fx0, fx1, fy0, fy1 = lims(city, .4)
ax.add_patch(plt.Rectangle((fx0, fy0), fx1 - fx0, fy1 - fy0, fill=False,
                           ec='k', lw=1.3, ls='--', zorder=8))
ax.annotate('主城放大区', (fx1, fy1), textcoords='offset points',
            xytext=(7, 6), ha='left', fontsize=9, zorder=9)
mark_center(ax)
for dn, g in ana[ana['距市中心_km'] > 12].groupby('所在区'):
    cx, cy = g['经度'].mean(), g['纬度'].mean()
    dy = -20 if cy > (y0 + y1) / 2 else 10      # 靠近上边界时标注下移，防裁切
    ax.annotate(f"{dn} (n={len(g)})", (cx, cy), textcoords='offset points',
                xytext=(9, dy), fontsize=8, color='dimgrey', zorder=9)
ax.set_title(f'① 全市概览 · {len(ana)} 个小区（虚线框 = 主城放大区）', fontsize=12)
ax.legend(fontsize=7.5, loc='lower left', framealpha=.9, markerscale=.7)
ax.grid(alpha=.2)

# -------- ② 主城放大 · 按区着色 --------
ax = axes[0, 1]
scatter_district(ax, city, 1.0)
ring(ax, 3); ring(ax, 6)
mark_center(ax)
set_extent(ax, city, .35)
ax.set_title(f'② 主城放大（≤{L_CITY:g}km，n={len(city)}）· 按区着色，点大小 = 设施总数',
             fontsize=12)
ax.legend(fontsize=7.5, loc='upper right', framealpha=.9, markerscale=.7)
ax.grid(alpha=.2)

# -------- ③ 主城放大 · 设施总数分异 --------
ax = axes[1, 0]
sc = ax.scatter(city['经度'], city['纬度'], c=city[TOTAL], s=54, cmap='RdYlGn',
                alpha=.9, edgecolors='grey', linewidths=.4, zorder=3)
ring(ax, 3); ring(ax, 6)
mark_center(ax)
set_extent(ax, city, .35)
plt.colorbar(sc, ax=ax, label='设施总数（个）', shrink=.9)
ax.set_title('③ 主城放大 · 500m 公共服务设施总数分异', fontsize=12)
ax.grid(alpha=.2)

# -------- ④ 核心区放大 · 绝对值最大者单独标注 --------
ax = axes[1, 1]
sc = ax.scatter(core['经度'], core['纬度'], c=core[TOTAL], s=80, cmap='RdYlGn',
                alpha=.9, edgecolors='grey', linewidths=.5, zorder=3)
ring(ax, 1); ring(ax, 2); ring(ax, 3)
mark_center(ax, label=False)
top3 = core.nlargest(3, TOTAL)
txt = '设施最多 TOP3\n' + '\n'.join(
    f"{i}. {r['小区名称']} {int(r[TOTAL])}" for i, (_, r) in enumerate(top3.iterrows()))
ax.annotate(txt, XJX, textcoords='offset points', xytext=(22, -58), fontsize=7.5,
            color='darkred', zorder=10,
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='darkred', alpha=.9),
            arrowprops=dict(arrowstyle='->', color='darkred', lw=.8))
set_extent(ax, core, .3)
plt.colorbar(sc, ax=ax, label='设施总数（个）', shrink=.9)
ax.set_title(f'④ 核心区放大（≤{L_CORE:g}km，n={len(core)}）', fontsize=12)
ax.grid(alpha=.2)

plt.savefig(os.path.join(OUT_DIR, '图2_空间分布.png'), dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ===== 4. 设施结构：占比 · 偏相关（控制密度）· 区际结构 =====
# 说明：6 类设施都是「同一 500m 圆内」的计数，共享潜变量「城市密度/区位」，
#       因此**原始两两相关**大部分是机械产物（PC1 ≈ 65% 方差），
#       已弃用；改为展示「控制密度后的偏相关」+「区际构成占比」。
from numpy.linalg import eigvalsh

print("① 六类设施占比（全样本 194 行）")
share = (ana[CATS].sum() / ana[CATS].sum().sum() * 100).round(1)
print(share.rename('占比%').to_string())

# ---------- 诊断：原始相关有多少只是「密度同步」----------
cm = ana[CATS].corr()                     # Pearson，与下方偏相关口径一致
iu = np.triu_indices(len(CATS), 1)
off = cm.values[iu]
evr = np.sort(eigvalsh(cm.values))[::-1]
evr = evr / evr.sum() * 100
print(f"\n② 原始两两相关：均值 {off.mean():.2f}，区间 [{off.min():.2f}, {off.max():.2f}]")
print(f"   相关矩阵 PC1 占 {evr[0]:.1f}% 方差 → 六类基本是同一个「密度」变量")

# ---------- 偏相关：控制密度代理（交通设施数）----------
def _resid(v, ctrl):
    """对 ctrl 做最小二乘回归后取残差（偏相关的标准做法）"""
    A = np.c_[np.ones(len(v)), ctrl]
    return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]

rest = [c for c in CATS if c != '交通设施数']
ctrl = ana['交通设施数'].values
pc = pd.DataFrame(index=rest, columns=rest, dtype=float)
for i in rest:
    ri = _resid(ana[i].values, ctrl)
    for j in rest:
        pc.loc[i, j] = 1.0 if i == j else np.corrcoef(ri, _resid(ana[j].values, ctrl))[0, 1]
pc = pc.astype(float)
off_pc = pc.values[np.triu_indices(len(rest), 1)]
print(f"\n③ 控制「交通设施数」后的偏相关：均值 {off_pc.mean():.2f}"
      f"（原 {off.mean():.2f}）→ 约 {100*(1-off_pc.mean()/off.mean()):.0f}% 的相关只是密度同步")
print(pc.round(2).to_string())
print("\n   真协同（偏相关显著为正）：")
for i, j in [('教育文化设施数', '体育休闲设施数'), ('教育文化设施数', '公园景点设施数'),
             ('公园景点设施数', '政务社区设施数')]:
    print(f"     {i[:-3]} × {j[:-3]}  ρ={pc.loc[i, j]:+.2f}")
print("   真替代（偏相关为负）：")
for i, j in [('公园景点设施数', '医疗卫生设施数'), ('政务社区设施数', '体育休闲设施数')]:
    print(f"     {i[:-3]} × {j[:-3]}  ρ={pc.loc[i, j]:+.2f}")

# ---------- 区际构成差异 ----------
comp = ana.groupby('所在区')[CATS].sum()
comp = (comp.div(comp.sum(axis=1), axis=0) * 100).round(1)
comp.columns = [c.replace('设施数', '') for c in CATS]
comp = comp.reindex([d_ for d_ in by_dist.index if d_ in comp.index])
print(f"\n④ 各区构成占比极差均值 {float((comp.max()-comp.min()).mean()):.1f} 个百分点"
      f" → 消去总量后，区际结构差异仍真实存在")
print(comp.to_string())

# ================= 图 3 =================
fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.9),
                         gridspec_kw={'width_ratios': [.85, 1, 1.2]})

# 3a 类别构成饼图
axes[0].pie(share.values, labels=[c.replace('设施数', '') for c in CATS],
            autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2', 6),
            textprops={'fontsize': 9},
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.2})
axes[0].set_title('① 500m 设施类别构成', fontsize=13)

# 3b 偏相关热力图（去掉对角线，突出非对角结构）
lbl = [c.replace('设施数', '') for c in rest]
mask = np.eye(len(rest), dtype=bool)
sns.heatmap(pc, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            mask=mask, square=True, ax=axes[1], annot_kws={'size': 8.5},
            cbar_kws={'shrink': .78, 'label': '偏相关 ρ'},
            xticklabels=lbl, yticklabels=lbl)
axes[1].set_title('② 控制「城市密度」后的偏相关\n（去同步后：红=真协同，蓝=真替代）', fontsize=12)
axes[1].tick_params(labelsize=8.5)

# 3c 区际构成热力图
sns.heatmap(comp, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[2],
            annot_kws={'size': 7.5}, linewidths=.6, linecolor='white',
            cbar_kws={'shrink': .78, 'label': '占区内设施总数 %'})
axes[2].set_title('③ 各区分项设施构成占比\n（行内归一化，已消去总量差异）', fontsize=12)
axes[2].tick_params(labelsize=8.5)
axes[2].set_xlabel(''); axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '图3_设施结构.png'), dpi=140, bbox_inches='tight')
plt.show()

print("\n⑤ 设施总数最多 / 最少的 10 个小区")
cols_view = ['小区名称', '所在区', TOTAL, '距市中心_km']
top = ana.nlargest(10, TOTAL)[cols_view].reset_index(drop=True)
bot = ana.nsmallest(10, TOTAL)[cols_view].reset_index(drop=True)
top['距市中心_km'] = top['距市中心_km'].round(1)
bot['距市中心_km'] = bot['距市中心_km'].round(1)
print("— 最多 —"); print(top.to_string())
print("\n— 最少 —"); print(bot.to_string())

In [ ]:
# ===== 4b 诊断：6 类设施之间的「相关性」是真发现，还是「城市密度」的重复？=====
# 关键：6 个变量都是**同一 500m 圆内**的计数，共享同一个潜变量（POI 密度/区位）。
#       若如此，高相关只是机械产物 → 热力图无信息量，需换图。
from scipy import stats as st
from numpy.linalg import eigvalsh

cm = ana[CATS].corr(method='spearman')
iu = np.triu_indices(len(CATS), 1)
off = cm.values[iu]

print("① 6 类设施两两 Spearman ρ")
print(cm.round(2).to_string())
print(f"\n两两相关：均值 {off.mean():.2f} | 中位 {np.median(off):.2f} | "
      f"最小 {off.min():.2f} | 最大 {off.max():.2f}")

# --- 主成分：若 6 类几乎共线，PC1 应吃掉绝大部分方差 ---
ev = np.sort(eigvalsh(cm.values))[::-1]
evr = ev / ev.sum() * 100
print(f"\n② 相关矩阵特征值占比：PC1 = {evr[0]:.1f}%   PC2 = {evr[1]:.1f}%   "
      f"PC3 = {evr[2]:.1f}%")
print(f"   → PC1 越高，越说明 6 类基本是**同一个变量**的不同切面")

# --- 偏相关：控制「交通设施数」(密度代理) 后还剩多少相关 ---
def resid(v, ctrl):
    A = np.c_[np.ones(len(v)), ctrl]
    return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]

rest = [c for c in CATS if c != '交通设施数']
pc = pd.DataFrame(index=rest, columns=rest, dtype=float)
for i in rest:
    for j in rest:
        pc.loc[i, j] = 1.0 if i == j else np.corrcoef(
            resid(ana[i].values, ana['交通设施数'].values),
            resid(ana[j].values, ana['交通设施数'].values))[0, 1]
print("\n③ 控制「交通设施数」后的偏相关 —— 剩下的才是真正的「配套协同」")
print(pc.round(2).to_string())
iu2 = np.triu_indices(len(rest), 1)
print(f"\n偏相关均值 {pc.values[iu2].mean():.2f}   （原相关均值 {off.mean():.2f}）")

# --- ④ 各区内部：类别的**构成占比**是否随区变化（这是真信息）---
print("\n④ 各区类别构成占比的离散度（占比 %  的极差，越大说明区际结构差异越明显）")
print((comp.max() - comp.min()).sort_values(ascending=False).round(1).to_string())
print(f"\n   占比极差均值 {(comp.max() - comp.min()).mean():.1f} 个百分点")


In [ ]:
# ===== 5. 区位：设施随距离的衰减规律 · 圈层差异 · 哪类配套掉得最快 =====
# 思路：设施总数随距市中心呈幂律衰减 → ① log-log 散点 + 拟合；
#       ② 五个距离圈层的总量分布（箱线图）；
#       ③ 六类设施在各圈层的平均数量（分组柱状）；
#       ④ 最外圈相对核心各类设施的"剩余比例"（哪类配套最先消失）。
x = np.log1p(ana['距市中心_km'])
y = np.log(ana[TOTAL])
b, a = np.polyfit(x, y, 1)
r2 = np.corrcoef(x, y)[0, 1] ** 2

print(f"① 幂律拟合：ln(设施数) = {a:.3f} + {b:.3f} × ln(1+距市中心km)")
print(f"   距市中心每翻一倍，设施数约降至 {2**b*100:.0f}%；R² = {r2:.3f}")

# ---------- 圈层统计 ----------
RING = ['<3km 核心', '3-6km', '6-10km', '10-20km', '>20km 外围']
ring = (ana.groupby('距市中心分层', observed=True)[TOTAL]
        .agg(小区数='size', 中位数='median', 均值='mean')
        .reindex(RING).round(0))
print("\n② 五个圈层的设施总数")
print(ring.to_string())

ring_mean = ana.groupby('距市中心分层', observed=True)[CATS].mean().reindex(RING)
core, outer = ring_mean.iloc[0], ring_mean.iloc[-1]
keep = (outer / core * 100).sort_values()
tbl = pd.DataFrame({'核心均值': core.round(0), '外围均值': outer.round(0),
                    '剩余比例%': (outer / core * 100).round(0)}).loc[keep.index]
tbl.index = [c.replace('设施数', '') for c in tbl.index]
print("\n③ 最外圈（>20km）相对核心（<3km）各类设施剩余比例")
print(tbl.to_string())

# ================= 图 4 =================
fig, axes = plt.subplots(2, 2, figsize=(15, 10.5))

# 5a：log-log 散点 + 幂律拟合（保留）
axes[0, 0].scatter(np.expm1(x), ana[TOTAL], s=26, alpha=.6, c='steelblue',
                   edgecolors='white', linewidths=.4)
xs = np.linspace(x.min(), x.max(), 50)
axes[0, 0].plot(np.expm1(xs), np.exp(a + b * xs), 'r--', lw=2,
                label=f'幂律基线 R²={r2:.2f}')
axes[0, 0].set_xscale('log'); axes[0, 0].set_yscale('log')
axes[0, 0].set_xlabel('距新街口距离（km，对数轴）')
axes[0, 0].set_ylabel('设施总数（个，对数轴）')
axes[0, 0].set_title('① 设施总数随距离呈幂律衰减', fontsize=13)
axes[0, 0].legend(); axes[0, 0].grid(alpha=.25, which='both')

# 5b：五个圈层的设施总数箱线图
sns.boxplot(data=ana, x='距市中心分层', y=TOTAL, order=RING, ax=axes[0, 1],
            hue='距市中心分层', palette='YlGnBu', legend=False, width=.6)
for i in range(len(RING)):
    med = ring['中位数'].iloc[i]
    axes[0, 1].text(i, med, f'{med:.0f}', va='center', fontsize=9, color='navy',
                    bbox=dict(facecolor='white', edgecolor='none', pad=.6, alpha=.75))
axes[0, 1].set_title('② 五个圈层对比：核心区中位 362，3km 外降至 150~220', fontsize=13)
axes[0, 1].set_xlabel('（10-20km 与 >20km 两档各仅 12 个小区，差异不稳定）')
axes[0, 1].set_ylabel('设施总数（个）')
axes[0, 1].set_xticklabels([f'{r}\nn={int(ring["小区数"].iloc[i])}'
                            for i, r in enumerate(RING)])
axes[0, 1].grid(axis='y', alpha=.3)

# 5c：六类设施 × 圈层平均数量
rm = ring_mean.rename(columns={c: c.replace('设施数', '') for c in CATS})
rm_t = rm.reset_index().melt(id_vars='距市中心分层',
                             var_name='类别', value_name='平均数量')
sns.barplot(data=rm_t, x='距市中心分层', y='平均数量', hue='类别',
            order=RING, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('③ 各类设施在各圈层的平均数量', fontsize=13)
axes[1, 0].set_xlabel(''); axes[1, 0].set_ylabel('平均设施数（个）')
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 0].legend(title='', fontsize=8, ncol=3)
axes[1, 0].grid(axis='y', alpha=.3)

# 5d：最外圈相对核心的剩余比例
kd = keep.rename(index={c: c.replace('设施数', '') for c in CATS})
cols = sns.color_palette('RdYlGn', len(kd))
axes[1, 1].barh(kd.index, kd.values, color=cols, edgecolor='white')
for i, v in enumerate(kd.values):
    axes[1, 1].text(v + 1.5, i, f'{v:.0f}%', va='center', fontsize=10)
axes[1, 1].set_xlim(0, 112)
axes[1, 1].set_title('④ 外围相对核心，各类配套还剩多少', fontsize=13)
axes[1, 1].set_xlabel('剩余比例（>20km 均值 ÷ <3km 均值）')
axes[1, 1].grid(axis='x', alpha=.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '图4_区位衰减与圈层差异.png'),
            dpi=140, bbox_inches='tight')
plt.show()

ana.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("\n已更新：分析底表.csv（含 距市中心_km）")


In [ ]:
# ===== 6. 极值小区：设施总数最多 / 最少的各 10 个 =====
# 目的：从 194 个小区中锁定配套两端，用「六类设施堆叠条形图」同时呈现
#       总量差距与结构差距——两端差多少、主要差在哪类设施。
# 口径：基于分析底表 ana；设施总数 = 六类设施数之和（先做一致性校验）。

# ---------- 口径校验与取数 ----------
gap = (ana[CATS].sum(axis=1) - ana[TOTAL]).abs()
print(f"① 口径校验：六类合计与设施总数最大偏差 {gap.max():.0f} → 堆叠分量严格等于总数\n")

top10 = ana.nlargest(10, TOTAL)
bot10 = ana.nsmallest(10, TOTAL)

def extreme_table(d, tag):
    t = d[['小区名称', '所在区', '计划年份', TOTAL, '距市中心_km']].copy()
    t.insert(0, '排名', range(1, len(t) + 1))
    t = t.rename(columns={TOTAL: '设施总数'})
    t['距市中心_km'] = t['距市中心_km'].round(1)
    print(f"② {tag}")
    print(t.to_string(index=False))

extreme_table(top10, '设施总数最多的 10 个小区')
print()
extreme_table(bot10, '设施总数最少的 10 个小区')

# ---------- 极值差距 / 区域 / 结构 ----------
mx, mn = ana[TOTAL].max(), ana[TOTAL].min()
name_max = ana.loc[ana[TOTAL].idxmax(), '小区名称']
name_min = ana.loc[ana[TOTAL].idxmin(), '小区名称']
print(f"\n③ 极值差距：{mx:.0f}（{name_max}）vs {mn:.0f}（{name_min}），相差 {mx / mn:.0f} 倍；"
      f"全样本中位 {ana[TOTAL].median():.0f}、均值 {ana[TOTAL].mean():.0f}")
print("   Top10 区分布：", top10['所在区'].value_counts().to_dict())
print("   Bottom10 区分布：", bot10['所在区'].value_counts().to_dict())
for tag, d in [('Top10', top10), ('Bottom10', bot10)]:
    sh = d[CATS].sum() / d[CATS].sum().sum() * 100
    txt = "，".join(f"{k.replace('设施数', '')}{v:.1f}%" for k, v in sh.items())
    print(f"   {tag} 构成：{txt}")

# ================= 图 5：六类堆叠横向条形图 =================
colors = sns.color_palette('Set2', 6)
fig, axes = plt.subplots(1, 2, figsize=(15.5, 7))

def draw_extreme(ax, d, title, mean_line=False):
    d = d.sort_values(TOTAL)                    # barh 自下而上 → 最大值置顶
    y = np.arange(len(d))
    left = np.zeros(len(d))
    for cat, c in zip(CATS, colors):
        ax.barh(y, d[cat], left=left, color=c, edgecolor='white',
                linewidth=.6, label=cat.replace('设施数', ''))
        left += d[cat].to_numpy()
    labels = [f"{r['小区名称']}\n{r['所在区']}·{r['计划年份']}批·距市中心{r['距市中心_km']:.1f}km"
              for _, r in d.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    for i, v in enumerate(d[TOTAL]):
        ax.text(v + d[TOTAL].max() * .01, i, f"{v:.0f}", va='center', fontsize=9)
    if mean_line:
        ax.axvline(ana[TOTAL].mean(), color='grey', ls='--', lw=1,
                   label=f"全样本均值 {ana[TOTAL].mean():.0f}")
    ax.set_xlim(0, d[TOTAL].max() * 1.13)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('设施数（个，按六类堆叠）')
    ax.grid(axis='x', alpha=.3)

draw_extreme(axes[0], top10,
             '设施总数最多的 10 个小区（均在老城核心、距新街口 ≤3km）', mean_line=True)
draw_extreme(axes[1], bot10,
             '设施总数最少的 10 个小区（外围新城为主，钟山风景区周边 3 个）')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(handles), loc='lower center',
           fontsize=9, frameon=False)
plt.tight_layout(rect=[0, .06, 1, 1])
plt.savefig(os.path.join(OUT_DIR, '图5_设施总数极值小区.png'),
            dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
# ===== 7. 综合分析：批次投向 · 配套盲区 · 高优先名单 · 配套模式聚类 =====
# 数据口径说明：溧水区高德设施数据不准确 → 依赖设施数的②④分析排除溧水（185 行）；
#               ① 批次投向只使用批次/区位原始信息，保留全部 194 行。
os.environ['OMP_NUM_THREADS'] = '1'
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RING = ['<3km 核心', '3-6km', '6-10km', '10-20km', '>20km 外围']
T = TOTAL
SHORT = {c: c.replace('设施数', '') for c in CATS}

# ---------- ① 改造批次投向 ----------
ct_ring = pd.crosstab(ana['距市中心分层'], ana['计划年份']).reindex(RING)
pct_ring = ct_ring.div(ct_ring.sum(axis=1), axis=0)
print("① 批次投向：各圈层 2024/2025 批数量")
print(ct_ring.to_string())
print("\n   各区 2025 批占比（升序）：")
ct_d = pd.crosstab(ana['所在区'], ana['计划年份'])
ct_d['2025占比%'] = (ct_d.get(2025, 0) / ct_d.sum(axis=1) * 100).round(0)
print(ct_d.sort_values('2025占比%')[['2025占比%']].T.to_string())

# ---------- ② 配套盲区（排除溧水）----------
ana_nl = ana[ana['所在区'] != '溧水区'].copy()
zero_cnt = (ana_nl[CATS] == 0).sum().sort_values(ascending=False)
cov = (ana_nl[CATS] > 0).sum(axis=1)
miss = ana_nl[cov < 6].copy()
miss['缺失类别'] = miss[CATS].apply(
    lambda r: '、'.join(SHORT[c] for c in CATS if r[c] == 0), axis=1)
print(f"\n② 配套盲区（已排除溧水，样本 {len(ana_nl)} 行）：六类不全的小区 {len(miss)} 个")
print("   各类设施 500m 内为 0 的小区数：", zero_cnt.rename(SHORT).to_dict())
print(miss[['小区名称', '所在区', '距市中心分层', '缺失类别']].to_string(index=False))

# ---------- ③ 高优先改造名单（主城 · 显著低于同区位基线）----------
# 残差在本单元内重算（图4单元未保留该列），保证自洽
_xv = np.log1p(ana['距市中心_km']); _yv = np.log(ana[T])
_bb, _aa = np.polyfit(_xv, _yv, 1)
ana['相对同区位偏离'] = _yv - (_aa + _bb * _xv)
hp = ana[(ana['距市中心_km'] <= 10) & (ana['相对同区位偏离'] < -0.6)].copy()
hp = hp.sort_values('相对同区位偏离')
print(f"\n③ 高优先改造名单：距新街口 ≤10km 且低于同区位基线 45%+ 的小区 {len(hp)} 个")
print(hp[['小区名称', '所在区', '距市中心_km', T]].round(1).to_string(index=False))

# ---------- ④ 配套模式聚类（排除溧水；另排除仅 1 类设施非零的极端点中山陵45号）----------
_cov4 = (ana_nl[CATS] > 0).sum(axis=1)
ana_cl = ana_nl[_cov4 >= 2].copy()
sh = ana_cl[CATS].div(ana_cl[T], axis=0)
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
lab = km.fit_predict(sh)
sil = silhouette_score(sh, lab)
centers = pd.DataFrame(km.cluster_centers_, columns=CATS)
centers = centers.div(centers.sum(axis=1), axis=0)        # 归一化为占比
sizes = pd.Series(lab).value_counts()
order_k = sizes.index.tolist()                            # 按规模排序展示
print(f"\n④ 配套模式聚类（K={K}，样本 {len(ana_cl)} 行：排除溧水及极端点中山陵45号，轮廓系数 {sil:.2f}）")
for k in order_k:
    row = centers.loc[k]
    top2 = row.sort_values(ascending=False).head(2)
    low1 = row.idxmin()
    desc = "、".join(f"{SHORT[c]}{v*100:.0f}%" for c, v in top2.items())
    print(f"   K{k} (n={sizes[k]}): 主导={desc}；最薄弱={SHORT[low1]} {row[low1]*100:.0f}%")
ana_cl['配套模式簇'] = [f'K{x}' for x in lab]
ana['配套模式簇'] = ana['小区名称'].map(
    ana_cl.set_index('小区名称')['配套模式簇'])
ana['配套模式簇'] = ana['配套模式簇'].fillna(
    pd.Series(np.where(ana['所在区'] == '溧水区', '未参与(溧水)', '未参与(极端点)'),
              index=ana.index))

# ================= 图 6 =================
colors = sns.color_palette('Set2', 6)
fig, axes = plt.subplots(2, 2, figsize=(15.5, 11))

# 6a：圈层 100% 堆叠（批次投向）
y = np.arange(len(RING))
axes[0, 0].barh(y, pct_ring[2024], color='#8FBBD9', label='2024 批')
axes[0, 0].barh(y, pct_ring[2025], left=pct_ring[2024], color='#E89B8A', label='2025 批')
for i in range(len(RING)):
    for yr, x0, c in [(2024, 0, '#2E5A78'), (2025, pct_ring[2024].iloc[i], '#8C3B2E')]:
        w = pct_ring[yr].iloc[i]
        if w > .12:
            axes[0, 0].text(x0 + w / 2, i, f'{yr}\n{int(ct_ring[yr].iloc[i])}个 {w*100:.0f}%',
                            ha='center', va='center', fontsize=8.5, color=c)
axes[0, 0].set_yticks(y); axes[0, 0].set_yticklabels(RING)
axes[0, 0].set_xlim(0, 1); axes[0, 0].set_xlabel('占比（%）')
from matplotlib.ticker import PercentFormatter
axes[0, 0].xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
axes[0, 0].set_title('① 批次投向：2024 兼顾外围，2025 聚焦主城', fontsize=13)
axes[0, 0].legend(loc='lower right'); axes[0, 0].grid(axis='x', alpha=.3)

# 6b：各类设施 0 值小区数
zc = zero_cnt.rename(index=SHORT)
axes[0, 1].bar(zc.index, zc.values, color=colors, edgecolor='white')
for i, v in enumerate(zc.values):
    axes[0, 1].text(i, v + .15, str(v), ha='center', fontsize=10)
axes[0, 1].set_title(f'② 配套盲区（排除溧水）：{len(miss)} 个小区六类不全', fontsize=13)
axes[0, 1].set_ylabel('500m 内该类设施为 0 的小区数')
axes[0, 1].grid(axis='y', alpha=.3)

# 6c：高优先名单
hp_y = np.arange(len(hp))
cols_hp = plt.cm.Reds(0.35 + 0.5 * (-hp['相对同区位偏离'] / -hp['相对同区位偏离'].min()))
axes[1, 0].barh(hp_y, hp['相对同区位偏离'], color=cols_hp, edgecolor='white')
axes[1, 0].set_yticks(hp_y)
axes[1, 0].set_yticklabels([f"{r['小区名称']}（{r['所在区']}）" for _, r in hp.iterrows()],
                           fontsize=8.5)
for i, (_, r) in enumerate(hp.iterrows()):
    axes[1, 0].text(r['相对同区位偏离'] - .02, i,
                    f"{r['相对同区位偏离']:.2f}｜{r['距市中心_km']:.1f}km｜共{r[T]:.0f}个",
                    va='center', ha='right', fontsize=7.8)
axes[1, 0].set_xlim(hp['相对同区位偏离'].min() * 1.28, 0)
axes[1, 0].set_xlabel('相对同区位偏离（越负越紧迫）')
axes[1, 0].set_title('③ 主城"地段好但配套差"的高优先小区', fontsize=13)
axes[1, 0].grid(axis='x', alpha=.3)

# 6d：簇心 100% 堆叠
yk = np.arange(K)
left = np.zeros(K)
for cat, c in zip(CATS, colors):
    vals = centers.loc[order_k, cat].to_numpy()
    axes[1, 1].barh(yk, vals, left=left, color=c, edgecolor='white', label=SHORT[cat])
    for i, v in enumerate(vals):
        if v > .07:
            axes[1, 1].text(left[i] + v / 2, i, f'{v*100:.0f}', ha='center',
                            va='center', fontsize=7.5)
    left += vals
axes[1, 1].set_yticks(yk)
axes[1, 1].set_yticklabels([f'K{k} (n={sizes[k]})' for k in order_k])
axes[1, 1].set_xlim(0, 1); axes[1, 1].set_xlabel('六类设施占比（%）')
axes[1, 1].xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
axes[1, 1].set_title('④ 四种典型配套模式的设施构成', fontsize=13)
axes[1, 1].legend(ncol=3, fontsize=8, loc='upper center')
axes[1, 1].grid(axis='x', alpha=.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '图6_批次盲区优先与配套模式.png'),
            dpi=140, bbox_inches='tight')
plt.show()

ana.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("\n已更新：分析底表.csv（新增 配套模式簇）")


In [ ]:
# ===== 8. 关键特征散点图矩阵（区位 × 六类设施）=====
# 读法：对角 = 各特征分布（按距中心圈层叠加）；上三角 = Spearman 秩相关 ρ；
#       下三角 = 散点（按圈层着色）+ 总体最小二乘趋势线（仅作方向示意）。
# 口径：8 个变量量纲差异大、强右偏（0~1273 个 / 0.19~49.6 km）→ 统一做 log1p 变换，
#       刻度仍回标原始单位；6 类设施同属一个 500m 圆内的计数，共享「城市密度」潜变量，
#       两两相关大部分是密度同步（同图 3 结论），故用秩相关 + 圈层着色把该驱动显式化。
from scipy import stats as st
from matplotlib.ticker import FuncFormatter

RING = ['<3km 核心', '3-6km', '6-10km', '10-20km', '>20km 外围']
SHORT = {c: c.replace('设施数', '') for c in CATS}
VARS = ['距市中心_km', TOTAL,
        '交通设施数', '政务社区设施数', '教育文化设施数',
        '医疗卫生设施数', '体育休闲设施数', '公园景点设施数']    # 六类按均值降序
NAME = {TOTAL: '设施总数', '距市中心_km': '距新街口(km)', **SHORT}
names = [NAME[v] for v in VARS]

# 绘图数据框：log1p 变换 + 圈层（PairGrid 的 hue 需要是列名，不能直接传 Series）
MD = ana[VARS].apply(np.log1p)
MD['圈层'] = pd.Categorical(ana['距市中心分层'].astype(str), categories=RING, ordered=True)
pal_ring = dict(zip(RING, sns.color_palette('YlGnBu', len(RING))))

# ---------- ① Spearman 秩相关 ----------
rho = pd.DataFrame(1.0, index=names, columns=names)
for i, u in enumerate(VARS):
    for v in VARS[i + 1:]:
        r = st.spearmanr(ana[u], ana[v]).statistic
        rho.loc[NAME[u], NAME[v]] = rho.loc[NAME[v], NAME[u]] = r
print("① Spearman 秩相关矩阵（对右偏稳健；下三角）")
print(rho.round(2).to_string())

print("\n② 区位驱动：距新街口 vs 各特征（越接近 0 = 越抗衰减）")
for k, v in rho.loc['距新街口(km)'].drop('距新街口(km)').sort_values().items():
    print(f"   {k:<10} ρ = {v:+.2f}")

offl = rho.loc[[NAME[c] for c in CATS], [NAME[c] for c in CATS]].values
offl = offl[np.triu_indices(len(CATS), 1)]
print(f"\n③ 六类之间：秩相关均值 {offl.mean():.2f}，区间 [{offl.min():.2f}, {offl.max():.2f}]")
print("   → 与图 3 的「PC1 ≈ 65% 方差」一致：同涨同落主要来自密度同步，非真实协同")

# ---------- 画矩阵 ----------
def _nice_ticks(v, k=5):
    """按原始量级挑 k 个整数刻度（返回原始值）"""
    mx = float(np.asarray(v).max())
    cand = [0, 1, 2, 5, 10, 20, 30, 50, 70, 100, 150, 200, 300, 500, 800, 1300]
    sel = [c for c in cand if c <= mx * 1.001]
    if len(sel) <= k:
        return sel
    idx = np.linspace(0, len(sel) - 1, k).round().astype(int)
    return sorted({sel[i] for i in idx})

def scatter_low(x, y, **kw):
    """下三角：散点 + 总体趋势线（仅示意方向，非拟合结论）"""
    ax = plt.gca()
    ax.scatter(x, y, s=14, alpha=.5, linewidths=0, color=kw.get('color'))
    b_, a_ = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 2)
    ax.plot(xs, a_ + b_ * xs, color='crimson', lw=1.1, ls='--', alpha=.8, zorder=5)

n = len(VARS)
g = sns.PairGrid(MD, vars=VARS, hue='圈层', hue_order=RING, palette=pal_ring,
                 height=2.05, diag_sharey=False)
g.map_diag(sns.histplot, element='step', fill=True, alpha=.35, bins=22, legend=False)
g.map_lower(scatter_low)

# 刻度回标原始单位（几何为 log1p）：对角显示本变量刻度，下三角只在底行/左列显示，上三角留白
fmt = FuncFormatter(lambda t, _: f'{np.expm1(t):,.0f}')
for i in range(n):
    for j in range(n):
        ax = g.axes[i, j]                      # 注意：PairGrid 为 [行=y变量, 列=x变量]
        ax.set_xticks([]); ax.set_yticks([])
        if i == j:                             # 对角：本变量的分布
            ax.set_xticks(np.log1p(_nice_ticks(ana[VARS[j]])))
            ax.xaxis.set_major_formatter(fmt)
        elif i > j:                            # 下三角：散点
            if i == n - 1:
                ax.set_xticks(np.log1p(_nice_ticks(ana[VARS[j]])))
                ax.xaxis.set_major_formatter(fmt)
            if j == 0:
                ax.set_yticks(np.log1p(_nice_ticks(ana[VARS[i]])))
                ax.yaxis.set_major_formatter(fmt)

# 上三角：只放 ρ 数值面板（i<j 即显示意义上的上三角）
for i in range(n):
    for j in range(i + 1, n):
        ax = g.axes[i, j]
        r = rho.iloc[i, j]
        ax.grid(False)
        ax.text(.5, .5, f'ρ = {r:+.2f}', transform=ax.transAxes, ha='center', va='center',
                fontsize=10.5, fontweight='bold' if abs(r) >= .6 else 'normal',
                color='crimson' if abs(r) >= .6 else 'dimgrey',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none', alpha=.85))

# 坐标轴标签（变量名 + 刻度说明）
for j in range(n):
    g.axes[n - 1, j].set_xlabel(f'{names[j]}', fontsize=10)
for i in range(n):
    g.axes[i, 0].set_ylabel(f'{names[i]}', fontsize=10)

g.figure.suptitle('关键特征散点图矩阵：区位 × 六类设施'
                  '（上三角 = Spearman ρ，下三角趋势线为总体拟合；刻度为 log 轴回标的原始值）',
                  fontsize=14, y=1.002)
g.add_legend(title='距新街口圈层', loc='upper right', frameon=True, fontsize=9,
             title_fontsize=10, markerscale=2.2)
g.figure.savefig(os.path.join(OUT_DIR, '图7_关键特征散点矩阵.png'),
                 dpi=140, bbox_inches='tight')
plt.show()

print("\n④ 读数提示：上三角 |ρ|≥0.60 用红色加粗——其中「设施总数 × 各类」是构造性相关"
      "（总数 = 六类之和），解读重点应放在「距新街口 × 各类」那一行：")
print("   公园景点、交通 衰减最快（ρ≈-0.65），医疗卫生 相对最抗衰减（ρ≈-0.41）")
print(f"   已保存：图7_关键特征散点矩阵.png（{len(ana)} 个小区 × {n} 个特征）")